# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$P(\text{dirección elegida})=0.90$$
$$P(\text{desviación izquierda})=0.05$$
$$P(\text{desviación derecha})=0.05$$

**Piso resbaloso**

$$P(\text{dirección elegida})=0.60$$
$$P(\text{desviación izquierda})=0.20$$
$$P(\text{desviación derecha})=0.20$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$\gamma=0.9,\qquad \theta=10^{-4}$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.

### Lectura de la imagen → coordenadas `(row, col)`

Recorriendo el grid de arriba-izquierda `(0,0)` hacia abajo-derecha `(4,5)`:

| Elemento | Celdas `(row, col)` |
|---|---|
| `START` | `(0,0)` |
| Estanterías / **paredes** | `(0,3)`, `(1,1)`, `(2,4)`, `(4,2)` |
| **Entrega** `+10` (terminal) | `(0,5)` |
| **Carga** `+2` (terminal) | `(2,2)` |
| **Peligro mortal** `-10` (terminal) | `(3,5)` |
| **Peligros** `-3` (no terminales) | `(1,4)`, `(4,1)` |
| **Piso resbaloso** (celdas amarillas) | `(1,2)`, `(2,1)`, `(3,3)` |


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self, living_reward=-1.0, gamma=0.9, slip_intended=0.60):
        self.height = 5
        self.width = 6

        # --- Traducción directa de la imagen (row, col) ---
        self.start = (0, 0)

        # Estanterías / paredes (celdas con estante gris)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Piso resbaloso (celdas amarillas)
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Estados TERMINALES con su recompensa R(s)
        self.terminal_states = {
            (0, 5): +10.0,   # zona de entrega
            (2, 2):  +2.0,   # estación de carga
            (3, 5): -10.0,   # peligro mortal
        }

        # Peligros NO terminales (-3): el robot los atraviesa pagando el castigo
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = living_reward
        self.gamma = gamma

        # Dinámica del piso (parametrizada para los experimentos A/B/C)
        self.p_intended_normal = 0.90
        self.p_perp_normal     = 0.05
        self.p_intended_slip   = slip_intended
        self.p_perp_slip       = (1.0 - slip_intended) / 2.0

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if self.is_valid_state((r, c))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): recompensa asociada al estado s.
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]   # -3 (no terminal)
        return self.living_reward              # costo por paso

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Un estado terminal es absorbente.
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Probabilidades según el tipo de piso del ESTADO ACTUAL.
        if state in self.slippery_states:
            p_int, p_perp = self.p_intended_slip, self.p_perp_slip
        else:
            p_int, p_perp = self.p_intended_normal, self.p_perp_normal

        # Desviaciones perpendiculares a la dirección elegida.
        perp1 = ( action[1],  action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_int),
            (perp1,  p_perp),
            (perp2,  p_perp),
        ]

        # Acumulamos probabilidades por estado destino (si choca pared/borde -> s' = s).
        agg = {}
        for move, prob in outcomes:
            ns = (state[0] + move[0], state[1] + move[1])
            if not self.is_valid_state(ns):
                ns = state
            agg[ns] = agg.get(ns, 0.0) + prob

        return list(agg.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # Σ_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # V_0(s) = 0
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()          # actualización sincrónica (todo con V_k)
        biggest_change = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)          # terminal absorbente
            else:
                best_q = max(
                    expected_next_value(grid, s, a, V)  # Σ T(s,a,s') V_k(s')
                    for a in grid.actions
                )
                # V_{k+1}(s) = R(s) + gamma * max_a Σ T V_k
                V_new[s] = grid.get_reward(s) + grid.gamma * best_q

            biggest_change = max(biggest_change, abs(V_new[s] - V[s]))

        V = V_new
        if biggest_change < threshold:   # criterio de convergencia theta
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a Σ_{s'} T(s,a,s') V(s')
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, s, a, V),
        )
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$


In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for _ in range(max_iter):
        delta = 0.0
        V_new = V.copy()

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                # Sin max: la política fija la acción pi(s).
                V_new[s] = (
                    grid.get_reward(s)
                    + grid.gamma * expected_next_value(grid, s, policy[s], V)
                )
            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    # pi_new(s) = argmax_a Σ T(s,a,s') V(s')
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, s, a, V),
        )
    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria (siempre UP)
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []   # nº de estados que cambian de acción en cada iteración

    for _ in range(max_iter):
        # 2. evaluación
        V = policy_evaluation(grid, policy, threshold)
        # 3. mejora
        new_policy = policy_improvement(grid, V)
        # 4. ¿estable?
        changed = sum(1 for s in policy if policy[s] != new_policy[s])
        history.append(changed)
        policy = new_policy
        if changed == 0:
            break

    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "\u2191",
    ( 1, 0): "\u2193",
    ( 0,-1): "\u2190",
    ( 0, 1): "\u2192",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")
        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n\u2713 Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [16, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas (configuración base: `living_reward=-1`, `gamma=0.9`)

**1. ¿El robot busca `+10` o `+2`?**
Con la configuración base (`living_reward=-1`, `gamma=0.9`), si seguimos la política *greedy* desde `START (0,0)` el robot termina en la **estación de carga `+2` en `(2,2)`**, no en la entrega `+10`. La razón es que llegar a `+10` queda lejos y exige pagar muchos `-1` por el camino: con `gamma=0.9` esa recompensa distante se descuenta tanto que el valor neto de buscar `+10` desde `START` queda por debajo del de alcanzar la carga `+2` cercana. (El barrido del **Bonus** confirma el umbral: para costos por paso de hasta ≈ `-0.7` la meta óptima es `+10`, y a partir de ≈ `-0.8` pasa a ser `+2`.) Nota: la primera acción impresa en `START` es `→`, pero el destino se decide siguiendo toda la trayectoria, no solo esa flecha.

**2. ¿Por qué una recompensa menor podría ser óptima?**
Porque lo que se maximiza es el **retorno descontado**, no la recompensa de un solo estado. Como llegar a `+10` exige muchos pasos costosos (`-1` cada uno) y la recompensa se descuenta con `gamma`, el valor descontado de ese camino largo resulta **menor** que ir a una meta más modesta `+2` pero cercana y barata de alcanzar. La optimalidad compara `R(s') + γ·(valor futuro)`, no solo `R(s')`; por eso aquí la meta "pequeña" `+2` termina siendo la óptima desde `START`.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**
En las celdas resbaladizas `(1,2)`, `(2,1)` y `(3,3)` —y sus vecinas— la acción elegida tiende a **alejarse de los estados peligrosos** (`-10` en `(3,5)`, `-3` en `(1,4)` y `(4,1)`). Como en piso resbaloso solo el 60% de las veces el robot va a donde quiere y el 40% se desvía, el algoritmo prefiere acciones cuyas desviaciones **no** caigan en un estado muy negativo. Es el mismo fenómeno del gridworld de clase: cerca de un acantilado la política resbaladiza se vuelve conservadora.

**4. ¿Qué papel cumple el costo por paso `-1`?**
Es un incentivo a **terminar rápido**. Hace que cada paso tenga un costo, de modo que caminos largos pierden valor y el robot prefiere rutas cortas hacia una meta positiva. Si fuera `0`, al robot le daría igual dar rodeos; si es muy negativo, puede preferir la meta cercana `+2` o incluso aceptar un final malo con tal de dejar de pagar el costo.

**5. ¿Por qué `T(s,a,s')` ya no puede usar las mismas probabilidades para todos los estados?**
Porque la dinámica **depende del tipo de piso del estado actual**: en piso normal la acción deseada ocurre con `0.90` y en piso resbaloso con `0.60`. Por tanto `get_transition_probs` debe mirar si `state` está en `slippery_states` y elegir el reparto de probabilidad correspondiente; no es un único modelo de ruido global.


### Experimento A — Menos costo por paso (`living_reward = -0.1`)

**Predicción:** con un costo por paso casi nulo, caminar deja de "doler". El robot debería dejar de conformarse con la carga `+2` y, desde cualquier punto, preferir viajar hasta la **entrega `+10`**, aun si eso implica más pasos. Los valores `V*` de casi todos los estados deberían subir (volverse menos negativos / más positivos).

In [7]:
grid_A = WarehouseMDP(living_reward=-0.1, gamma=0.9)
V_A, n_A = value_iteration(grid_A)
pi_A = extract_policy(grid_A, V_A)
print(f"Experimento A — living_reward=-0.1  (iteraciones: {n_A})")
print("\nValores:")
print_values(grid_A, V_A)
print("\nPolítica:")
print_policy(grid_A, pi_A)
print("\nAcción desde START:", ARROWS[pi_A[grid_A.start]])


Experimento A — living_reward=-0.1  (iteraciones: 26)

Valores:
 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Acción desde START: →


**Observación:** al abaratar el paso, los valores suben respecto a la base (muchos se vuelven positivos) y conviene recorrer más distancia hacia `+10`. Esto es coherente con el Bonus: con un costo por paso tan bajo (`-0.1`), el destino óptimo desde la zona del `START` es la entrega `+10`.

### Experimento B — Piso muy resbaloso (`0.60 → 0.40`)

Reparto en piso resbaloso: deseado `0.40`, cada desviación `0.30`.

**Predicción:** con más resbalamiento, cruzar o bordear celdas resbaladizas se vuelve más arriesgado (mayor probabilidad de desviarse hacia peligros). Esperamos sobre todo una **caída de los valores** en los estados resbaladizos y sus vecinos; la *política* puede o no cambiar de flecha, porque la acción que ya apuntaba lejos del peligro sigue siendo la mejor aunque el riesgo aumente.

In [8]:
grid_B = WarehouseMDP(living_reward=-1.0, gamma=0.9, slip_intended=0.40)
V_B, n_B = value_iteration(grid_B)
pi_B = extract_policy(grid_B, V_B)
print(f"Experimento B — slippery intended=0.40 (perp=0.30 c/u)  (iteraciones: {n_B})")
print("\nValores:")
print_values(grid_B, V_B)
print("\nPolítica:")
print_policy(grid_B, pi_B)
print("\nAcción desde START:", ARROWS[pi_B[grid_B.start]])

# Comparación directa con la base en los estados resbaladizos
print("\nComparación de acciones en celdas resbaladizas (base -> B):")
for s in sorted(grid.slippery_states):
    print(f"  {s}: {ARROWS[pi_vi[s]]}  ->  {ARROWS[pi_B[s]]}")


Experimento B — slippery intended=0.40 (perp=0.30 c/u)  (iteraciones: 22)

Valores:
 -2.706 |  -1.809 |  -0.797 |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.395 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Acción desde START: →

Comparación de acciones en celdas resbaladizas (base -> B):
  (1, 2): ↓  ->  ↓
  (2, 1): →  ->  →
  (3, 3): ↑  ->  ↑


### Experimento C — Más paciencia (`gamma = 0.99`)

**Predicción:** con `γ=0.99` el futuro se descuenta menos, así que recompensas lejanas pesan más. El robot debería valorar aún más la **entrega `+10` distante** y los valores de los estados lejanos a la meta deberían aumentar notablemente respecto a la base.

In [9]:
grid_C = WarehouseMDP(living_reward=-1.0, gamma=0.99)
V_C, n_C = value_iteration(grid_C)
pi_C = extract_policy(grid_C, V_C)
print(f"Experimento C — gamma=0.99  (iteraciones: {n_C})")
print("\nValores:")
print_values(grid_C, V_C)
print("\nPolítica:")
print_policy(grid_C, pi_C)
print("\nAcción desde START:", ARROWS[pi_C[grid_C.start]])
print(f"\nV*(START):  base(gamma=0.9) = {V_vi[grid.start]:+.3f}   |   C(gamma=0.99) = {V_C[grid_C.start]:+.3f}")


Experimento C — gamma=0.99  (iteraciones: 24)

Valores:
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Acción desde START: →

V*(START):  base(gamma=0.9) = -2.575   |   C(gamma=0.99) = -1.456


**Observación:** al subir `γ`, `V*(START)` aumenta: la recompensa `+10` lejana se descuenta menos y el robot está más dispuesto a 'invertir' pasos para alcanzarla.

### Bonus — ¿Para qué `living_reward` cambia la decisión desde `START`?

Barremos `living_reward` y detectamos el valor umbral donde la **ruta óptima desde `START`** deja de dirigirse a la entrega `+10` y pasa a preferir la carga `+2` (o viceversa). Para clasificar el destino seguimos la política *greedy* (acción más probable) desde `START` hasta caer en un terminal.

In [10]:
def greedy_terminal_from_start(grid, policy):
    """Sigue la acción más probable desde START hasta un terminal (destino 'intencional')."""
    s = grid.start
    for _ in range(200):
        if grid.is_terminal(s):
            return s
        a = policy[s]
        # estado destino más probable bajo esa acción
        ns = max(grid.get_transition_probs(s, a), key=lambda t: t[1])[0]
        if ns == s:          # chocó contra pared/borde -> evita bucle infinito
            return None
        s = ns
    return None

print(f"{'living_reward':>14} | {'START':>6} | destino (greedy)")
print("-" * 48)
umbral_prev = None
dest_prev = None
for lr in [round(-x, 3) for x in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9,
                                   1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.5, 3.0, 4.0, 5.0]]:
    g = WarehouseMDP(living_reward=lr, gamma=0.9)
    V, _ = value_iteration(g)
    pol = extract_policy(g, V)
    dest = greedy_terminal_from_start(g, pol)
    etiqueta = {(0,5): "ENTREGA +10", (2,2): "CARGA +2", (3,5): "MUERTE -10"}.get(dest, str(dest))
    print(f"{lr:>14} | {ARROWS[pol[g.start]]:>6} | {etiqueta}")


 living_reward |  START | destino (greedy)
------------------------------------------------
          -0.0 |      → | ENTREGA +10
          -0.1 |      → | ENTREGA +10
          -0.2 |      → | ENTREGA +10
          -0.3 |      → | ENTREGA +10
          -0.4 |      → | ENTREGA +10
          -0.5 |      → | ENTREGA +10
          -0.6 |      → | ENTREGA +10
          -0.7 |      → | ENTREGA +10
          -0.8 |      → | CARGA +2
          -0.9 |      → | CARGA +2
          -1.0 |      → | CARGA +2
          -1.2 |      → | CARGA +2
          -1.4 |      → | CARGA +2
          -1.6 |      → | CARGA +2
          -1.8 |      → | CARGA +2
          -2.0 |      → | CARGA +2
          -2.5 |      → | CARGA +2
          -3.0 |      → | CARGA +2
          -4.0 |      → | CARGA +2
          -5.0 |      → | CARGA +2


**Interpretación del bonus:** el barrido muestra el punto donde la política desde `START` cambia de destino. Mientras el costo por paso es bajo, al robot le conviene invertir pasos y buscar la **entrega `+10`**; a medida que el paso se vuelve más caro, llega un umbral de `living_reward` a partir del cual la meta cercana **carga `+2`** (menos pasos, menos castigo acumulado) pasa a ser óptima. El valor exacto se lee en la tabla anterior como la fila donde cambia la columna *destino*.